In [ ]:
import os
import sys
import ast
import pandas as pd

In [ ]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [ ]:
from llm_models.code_llms import Mistral
from code_inconsistency.code_inconsistency_tester import LLMConsistencyTester
from code_inconsistency.prompt_templates.prompt_template import CodeInconsistencyPromptTemplate

In [ ]:
class NewMutationPreConditionFilters:
    """
    Helper class to filter database entries that contain code constructs 
    suitable for the new mutation types.
    """
    
    @staticmethod
    def has_boolean_literals(code_str: str) -> bool:
        """
        Check if the code contains boolean literals (True/False) that can be mutated.
        
        Args:
            code_str (str): The code to analyze
            
        Returns:
            bool: True if code contains True or False literals
        """
        try:
            tree = ast.parse(code_str)
            
            # Walk through all nodes to find boolean constants
            for node in ast.walk(tree):
                # Check for Constant nodes with boolean values (Python 3.8+)
                if isinstance(node, ast.Constant) and isinstance(node.value, bool):
                    return True
                # Check for NameConstant nodes (older Python versions)
                if hasattr(ast, 'NameConstant') and isinstance(node, ast.NameConstant):
                    if node.value in [True, False]:
                        return True
                
            return False
            
        except (SyntaxError, ValueError):
            return False
    
    @staticmethod
    def has_unfoldable_constants(code_str: str) -> bool:
        """
        Check if the code contains integer constants in the range 4-15 that can be unfolded.
        
        Args:
            code_str (str): The code to analyze
            
        Returns:
            bool: True if code contains integer constants suitable for unfolding
        """
        try:
            tree = ast.parse(code_str)
            
            # Walk through all nodes to find integer constants
            for node in ast.walk(tree):
                # Check for Constant nodes with integer values (Python 3.8+)
                if isinstance(node, ast.Constant) and isinstance(node.value, int):
                    if 4 <= node.value <= 15:
                        return True
                # Check for Num nodes (older Python versions)
                if hasattr(ast, 'Num') and isinstance(node, ast.Num):
                    if isinstance(node.n, int) and 4 <= node.n <= 15:
                        return True
                
            return False
            
        except (SyntaxError, ValueError):
            return False
    
    @staticmethod
    def has_commutative_operations(code_str: str) -> bool:
        """
        Check if the code contains addition or multiplication operations that can be reordered.
        
        Args:
            code_str (str): The code to analyze
            
        Returns:
            bool: True if code contains + or * operations
        """
        try:
            tree = ast.parse(code_str)
            
            # Walk through all nodes to find binary operations
            for node in ast.walk(tree):
                # Check for BinOp nodes with Add or Mult operations
                if isinstance(node, ast.BinOp):
                    if isinstance(node.op, (ast.Add, ast.Mult)):
                        return True
                
            return False
            
        except (SyntaxError, ValueError):
            return False
    
    @staticmethod
    def filter_database_for_mutation(llmtester, mutation_type: str):
        """
        Filter the database to only include entries suitable for the specified mutation type.
        
        Args:
            llmtester: LLMConsistencyTester instance
            mutation_type (str): Type of mutation to filter for
            
        Returns:
            list: List of filtered document IDs suitable for the mutation
        """
        suitable_docs = []
        
        # Get all documents from the database
        all_docs = list(llmtester.question_database.find({}))
        
        print(f"Checking {len(all_docs)} documents for {mutation_type} mutation suitability...")
        
        # Choose the appropriate filter function
        if mutation_type == "boolean_literal":
            filter_func = NewMutationPreConditionFilters.has_boolean_literals
        elif mutation_type == "constant_unfold":
            filter_func = NewMutationPreConditionFilters.has_unfoldable_constants
        elif mutation_type == "commutative_reorder":
            filter_func = NewMutationPreConditionFilters.has_commutative_operations
        else:
            raise ValueError(f"Unknown mutation type: {mutation_type}")
        
        for doc in all_docs:
            # Check if the solution contains the target constructs
            if 'full_sol' in doc and filter_func(doc['full_sol']):
                suitable_docs.append(doc['_id'])
        
        print(f"Found {len(suitable_docs)} documents suitable for {mutation_type} mutation out of {len(all_docs)} total documents")
        
        return suitable_docs

In [ ]:
# Initialize the tester and LLM
llmtester = LLMConsistencyTester("HumanEval_Input_Output")
llm = Mistral()

# Test basic database connection
print(f"Database connection established")
total_docs = llmtester.question_database.count_documents({})
print(f"Total documents in database: {total_docs}")

# Show a sample document to verify structure
sample_doc = llmtester.question_database.find_one({})
if sample_doc:
    print(f"\nSample document structure:")
    print(f"Document ID: {sample_doc['_id']}")
    print(f"Has 'full_sol' field: {'full_sol' in sample_doc}")
    if 'full_sol' in sample_doc:
        print(f"Solution preview: {sample_doc['full_sol'][:100]}...")
else:
    print("No documents found in database")

In [ ]:
# Set up results directory
model_name = "mistral-small-2506"
mistral_results = os.path.join(proj_dir + '/results/code_inconsistencies/mistral')
os.makedirs(mistral_results, exist_ok=True)
print(f"Results will be saved to: {mistral_results}")

## Testing Boolean Literal Mutations

First, let's test the `boolean_literal` mutation type which transforms:
- `True` ↔ `not False`  
- `False` ↔ `not True`

In [ ]:
# Filter documents suitable for boolean_literal mutations
boolean_literal_docs = NewMutationPreConditionFilters.filter_database_for_mutation(llmtester, "boolean_literal")
print(f"\nDocuments suitable for boolean_literal mutation: {len(boolean_literal_docs)}")
if len(boolean_literal_docs) > 0:
    print(f"First 10 document IDs: {boolean_literal_docs[:10]}")

In [ ]:
'''
# Run No-Mutation baseline test for boolean_literal suitable documents
if len(boolean_literal_docs) > 0:
    # Set up file naming for baseline
    syntactic_mutation = None
    prompt_type = "zero_shot"
    mutation_str = f"boolean_literal_baseline"
    
    output_file_path = f"{mistral_results}/{model_name}_{prompt_type}_{mutation_str}.csv"
    print(f"Baseline results will be saved to: {output_file_path}")
    
    # Test first 10 documents
    test_docs = boolean_literal_docs
    print(f"Running No-Mutation baseline test on {len(test_docs)} suitable documents")
    
    # Run baseline test (no mutation)
    baseline_pass_count = llmtester._run_code_consistency_test(
        llm=llm,
        prompt_helper=CodeInconsistencyPromptTemplate.OutputPrediction.zero_shot_prompt,
        prompt_type=prompt_type,
        output_file_path=output_file_path,
        specific_doc_ids=test_docs,
        syntactic_mutation=syntactic_mutation
    )
    
    print(f"\nBoolean Literal Baseline test completed!")
    print(f"Processed: {len(test_docs)} documents")
    print(f"Results saved to: {output_file_path}")
else:
    print("No documents found suitable for boolean_literal mutation. Skipping baseline test.")
    '''

In [ ]:

# Run Boolean Literal mutation test on filtered documents
if len(boolean_literal_docs) > 0:
    # Set up file naming for mutation test
    syntactic_mutation = "boolean_literal"
    prompt_type = "zero_shot"
    mutation_str = f"{syntactic_mutation}_mutation"
    
    output_file_path = f"{mistral_results}/{model_name}_{prompt_type}_{mutation_str}.csv"
    print(f"Mutation results will be saved to: {output_file_path}")
    
    # Test same documents as baseline
    test_docs = boolean_literal_docs
    print(f"Running Boolean Literal mutation test on {len(test_docs)} suitable documents")
    
    # Run mutation test
    mutation_pass_count = llmtester._run_code_consistency_test(
        llm=llm,
        prompt_helper=CodeInconsistencyPromptTemplate.OutputPrediction.zero_shot_prompt,
        prompt_type=prompt_type,
        output_file_path=output_file_path,
        specific_doc_ids=test_docs,
        syntactic_mutation=syntactic_mutation
    )
    
    print(f"\nBoolean Literal mutation test completed!")
    print(f"Processed: {len(test_docs)} documents")
    print(f"Results saved to: {output_file_path}")
else:
    print("No documents found suitable for boolean_literal mutation. Skipping mutation test.")

    

## Analysis Notes - Boolean Literal Mutations

The `boolean_literal` mutation transforms boolean constants:
- `True` becomes `not False`
- `False` becomes `not True`

These transformations are semantically equivalent but syntactically different, making them good tests for LLM consistency on boolean logic understanding.